In [ ]:
# 필요 모듈 import
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import pandas as pd

# read data
df = pd.read_csv("customer_data.csv")
# 분석에 사용될 주요 features 선택
features = ['Income', 'Spent', 'Age', 'Recency']  

# null값 제거
df = df.dropna(subset=features)

# scaling 사전 처리 필요
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df[features])

# 
# 1. kmeans 클러스터링
kmeans = KMeans(n_clusters=4, random_state=42)
df['KMeans_Cluster'] = kmeans.fit_predict(scaled_data)

# kmeans 실루엣 score evaluation
kmeans_silhouette = silhouette_score(scaled_data, df['KMeans_Cluster'])
print(f"KMeans Silhouette Score: {kmeans_silhouette:.2f}")

# 
# 2. dbscan 클러스터링
dbscan = DBSCAN(eps=1, min_samples=10) # 두 수치는 분석 결과를 보고 적당히 조정
df['DBSCAN_Cluster'] = dbscan.fit_predict(scaled_data)

# dbscan 실루엣 score evaluation
valid_idx = df['DBSCAN_Cluster'] != -1  # 군집이 없는 이상치 제거
dbscan_silhouette = silhouette_score(scaled_data[valid_idx], df.loc[valid_idx, 'DBSCAN_Cluster'])
print(f"DBSCAN Silhouette Score: {dbscan_silhouette:.2f}")


- kmeans의 경우 해석이 쉽고, 수치형 데이터에 적합하며 원하는 cluster 수를 지정할 수 있음
- dbscan의 경우 cluster 수를 지정할 수 없고 자동으로 밀도기반으로 분류되며, 이상치를 제거하기 용이함.
- 따라서 타겟 고객을 쇼핑몰 직원들이 원하는 정도의 class로 구분하여 분석하기 쉽도록 kmeans를 사용하는 것이 적절할 것으로 보임.

In [ ]:
 (4) 클러스터 결과 개선 방안
[현 시각화 결과 분석]

클러스터 0: 전체적으로 가장 밀집되어 있으며, 중간 소득 & 중간 소비 고객 다수 포함

클러스터 1: 중소득 & 고소비 고객 일부 존재, 경계가 불분명

클러스터 2: 고소득 & 중소비 이상 고객 분포, 일부 이상치 포함

클러스터 3: 저소득 & 저소비 고객, 밀집되어 있으나 분산이 적음

[개선 필요성]

일부 클러스터 간 경계가 불명확하며 소득이 높은 이상치가 포함된 클러스터가 존재

클러스터 0과 1은 소비 지출에서 차이가 있으나 시각적으로 겹침 → 분리가 명확하지 않음

클러스터 수 (n_clusters=4)가 현재 데이터 특성을 완전히 반영하지 못하는 가능성 있음

✅ 개선 방안 제안


문제점	개선 방안
클러스터 간 경계 불명확	- PCA 또는 t-SNE, UMAP 등으로 고차원 변수 포함 후 시각화
- 스케일 조정 재확인 (StandardScaler / RobustScaler)
이상치 영향	- 이상치 제거 후 클러스터링 재수행
- 또는 DBSCAN 등 이상치에 강한 알고리즘 사용 고려
정보 부족	- 현재 Income과 Spent만 사용됨 → Age, Recency 등 추가 변수 포함
군집 수 부적절	- 엘보우 기법 / 실루엣 점수를 통해 최적 k 탐색
🔹 (5) 기획자 대상 요약 보고 (고객 세그먼트 구성 설명)
비전문가(기획자)를 위한 간단한 요약문은 다음과 같이 제시할 수 있습니다:

고객 세그먼트 분석 결과 요약
이번 저희 쇼핑몰 고객 데이터 분석을 통해 고객을 총 4개의 그룹으로 분류하였습니다.
각 그룹은 '소득'과 '소비' 값에 따라 다음과 같은 특징을 가집니다:

그룹 0 (소득 中, 소비 中)
- 가장 많은 고객이 해당하는 그룹
- 전체 고객의 중심이므로, 소비를 유지할 수 있도록 하는 정책 유리

그룹 1 (소득 上, 소비 上)
- 그룹 1보다 수는 적으나, 소비량이 더 큼.
- 마찬가지로 소비를 유지할 수 있도록 하며, 프리미엄 서비스 도입 시도 가능

그룹 2 (소득 下, 소비 中下)
- 소득도 지출도 높지 않지만, 소득에 비해 지출이 있는 편
- 가성비 마케팅 유리

그룹 3 (소득 中, 소비 下)
- 소득 무관, 지출이 적음
- 할인 프로모션 또는 신규 고객 전환 유도 유리
